In [18]:
from faster_whisper import WhisperModel

In [20]:
small = WhisperModel("small.en", device="cpu", compute_type="int8")


In [21]:
tiny = WhisperModel("tiny.en", device="cpu", compute_type="int8")

In [22]:
base = WhisperModel("base.en", device="cpu", compute_type="int8")

In [9]:
from IPython.display import Audio , display

In [ ]:
from glob import glob
test_data = glob('final_test_audios/mayan/Standard recording *.wav')
for i in test_data:
    display(Audio(i))
    segments, info = model.transcribe(i, beam_size=5)

    for segment in segments:
        print(segment.text)

    print("\n\n")

In [23]:
import unicodedata 

def remove_accents(text):
    nfkd_form = unicodedata.normalize('NFKD', text)
    return "".join([c for c in nfkd_form if not unicodedata.combining(c)])

In [24]:
## text preprocessing

def text_preprocessing(text):
    text = text.lower().strip()
    text = contractions.fix(text)
    text = remove_accents(text)
    def convert_decimal(match):
        number = match.group(0)
        integer, decimal = number.split(".")
        integer_words = num2words(int(integer))
        decimal_words = num2words(int(decimal))
        return f"{integer_words} point {decimal_words}"
        
    text = re.sub(r"\b\d+\.\d+\b", convert_decimal, text)
    text = re.sub(r"\b\d+\b", lambda x: num2words(int(x.group(0))), text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.split()


In [25]:
## sequance matching 

def sequance_matching_score(target_tokens, vosk_tokens):

    matcher = difflib.SequenceMatcher(None, target_tokens, vosk_tokens)
    
    report = []
    correct_words_count = 0
    total_teacher_words = len(target_tokens)

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        
        if tag == 'equal':
            chunk_len = i2 - i1
            correct_words_count += chunk_len
            
            for i in range(i1, i2):
                report.append({
                    "word": target_tokens[i],
                    "status": "match"
                })

        elif tag == 'delete':
            for i in range(i1, i2):
                report.append({
                    "word": target_tokens[i],
                    "status": "Missed"
                })

        elif tag == 'replace':

            teacher_chunk = target_tokens[i1:i2]
            vosk_chunk = vosk_tokens[j1:j2]
            
            for t_word, v_word in zip_longest(teacher_chunk, vosk_chunk, fillvalue=None):
                
                if t_word is None:
                    break  

                if v_word is None:
                    report.append({
                        "word": t_word,
                        "status": "Missed"
                    })
                    continue 

                else:
                    similarity = difflib.SequenceMatcher(None, t_word, v_word).ratio()
                    
                    if similarity >= 0.8:
                        correct_words_count += 1
                        report.append({
                            "word": t_word, 
                            "status": f"Accepted typo ({int(similarity*100)}%)"
                        })
                    else:
                        report.append({
                            "word": t_word, 
                            "status": f"Wrong word. Heard '{v_word}'"
                        })

    if total_teacher_words == 0:
        final_score = 0
    else:
        final_score = int((correct_words_count / total_teacher_words) * 100)

    return final_score, report


In [26]:
import difflib
from itertools import zip_longest
import re
import contractions
from num2words import num2words

In [27]:
def check_fuzzy_keywords(child_speech, keywords, threshold=0.8):
    missing_words = []
    for target in keywords:
        found = False

  
        if target in child_speech:
            found = True
        else:
            for word in child_speech:
                similarity = difflib.SequenceMatcher(None, target, word).ratio()
                if similarity >= threshold:
                    found = True
                    break
        
        if not found:
            missing_words.append(target)

    if len(missing_words) == 0:
        return True
    else:
        return False
    


In [28]:
targets = [
    "red car",
    "One plus one is two.",
    "a square",             
    "1, 2, 3, 4, 5.",
    "This is my mom.",
    "I love my mom.",
    "Good morning. Good night.",
    "Sunday, Monday, Tuesday, Wednesday, Thursday, Friday.",
    "Under chair.",
    "love family.",
    "dog.",
    "Red.",
]

In [31]:
def audio_test(file_path, model):
    try:
        segments, info = model.transcribe(file_path, beam_size=5)
        text = " ".join([segment.text for segment in segments])
        return text if text else "" # Return empty string if text is None
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return "" # Return empty string on error

In [32]:
def audio_test_whisper(audio_path, model):
    # 1. Load Audio (Force 16kHz mono)
    # This ensures Whisper gets the exact same input array as Vosk
    audio_float32, _ = librosa.load(audio_path, sr=16000, mono=True)
    
    # 2. Pre-process (Whisper accepts float32 natively, so no conversion needed)
    
    # 3. Transcribe (Pass the numpy array, NOT the file path)
    segments, info = model.transcribe(audio_float32, beam_size=5)
    
    text = " ".join([segment.text for segment in segments])
    return text

Target:        ['red', 'car']
Whisper Tiny:  ['red', 'color']  (Time: 2.115s)
Whisper Base:  ['red', 'car']  (Time: 1.332s)
Whisper Small: ['red', 'car']  (Time: 3.859s)



fuzzy match
Tiny:  False
Base:  True
Small: True


_____________________________________


shadowing match
--- TINY REPORT ---
red             | match
car             | Wrong word. Heard 'color'
_____________________________________
--- BASE REPORT ---
red             | match
car             | match
_____________________________________
--- SMALL REPORT ---
red             | match
car             | match
_____________________________________
Scores -> Tiny: 50 | Base: 100 | Small: 100



__________________________________________________________________
Target:        ['one', 'plus', 'one', 'is', 'two']
Whisper Tiny:  ['one', 'plus', 'one', 'is', 'two']  (Time: 0.728s)
Whisper Base:  ['one', 'plus', 'one', 'is', 'two']  (Time: 1.394s)
Whisper Small: ['one', 'plus', 'one', 'is', 'two']  (Time: 3.857s)



fuzzy match


In [33]:
targets = [
    ("red car",'fuzzy'),
   ( "One plus one is two.", 'shadow'),
    ("a square",'fuzzy'),             
   ( "1, 2, 3, 4, 5.", 'shadow'),
   ( "This is my mom.", 'shadow'),
   ( "I love my mom.", 'shadow'),
    ("Good morning. Good night.", 'shadow'),
    ("Sunday, Monday, Tuesday, Wednesday, Thursday, Friday." ,'fuzzy'),
   ( "Under chair." ,'fuzzy'),
    ("love family." ,'fuzzy'),
    ("dog." ,'fuzzy'),
    ("Red." ,'fuzzy'),
]

In [35]:
import time 

tiny_acc_shadwing = []
tiny_acc_fuzzy = []
tiny_latency = [] 

base_acc_shadwing = []
base_acc_fuzzy = []
base_latency = [] 

small_acc_shadwing = []
small_acc_fuzzy = []
small_latency = [] 


for i, (target, ttype) in zip(test_data, targets):
    target_norm = text_preprocessing(target)
    
    
    # Test Tiny
    start = time.time()
    raw_tiny = audio_test(i, tiny)
    dur_tiny = time.time() - start
    tiny_latency.append(dur_tiny)
    tiny_norm = text_preprocessing(raw_tiny)

    # Test Base
    start = time.time()
    raw_base = audio_test(i, base)
    dur_base = time.time() - start
    base_latency.append(dur_base)
    base_norm = text_preprocessing(raw_base)

    # Test Small
    start = time.time()
    raw_small = audio_test(i, small)
    dur_small = time.time() - start
    small_latency.append(dur_small)
    small_norm = text_preprocessing(raw_small)
    
    print(f"Target:        {target_norm}")
    print(f"Whisper Tiny:  {tiny_norm}  (Time: {dur_tiny:.3f}s)")
    print(f"Whisper Base:  {base_norm}  (Time: {dur_base:.3f}s)")
    print(f"Whisper Small: {small_norm}  (Time: {dur_small:.3f}s)")
    print("\n\n")

    if ttype == "fuzzy":
        print('fuzzy match')
        passed_tiny = check_fuzzy_keywords(tiny_norm, target_norm)
        passed_base = check_fuzzy_keywords(base_norm, target_norm)
        passed_small = check_fuzzy_keywords(small_norm, target_norm)

        tiny_acc_fuzzy.append(passed_tiny)
        base_acc_fuzzy.append(passed_base)
        small_acc_fuzzy.append(passed_small)
    
        print(f"Tiny:  {passed_tiny}")
        print(f"Base:  {passed_base}")
        print(f"Small: {passed_small}")
        print("\n")
        print("_____________________________________")
        print("\n")
    elif ttype == "shadow":
        print("shadowing match")
        score_tiny, report_tiny = sequance_matching_score(target_norm, tiny_norm)
        score_base, report_base = sequance_matching_score(target_norm, base_norm)
        score_small, report_small = sequance_matching_score(target_norm, small_norm)

        tiny_acc_shadwing.append(score_tiny)
        base_acc_shadwing.append(score_base)
        small_acc_shadwing.append(score_small)

        print("--- TINY REPORT ---")
        for item in report_tiny:
            print(f"{item['word']:<15} | {item['status']}")
        print("_____________________________________")
    
        print("--- BASE REPORT ---")
        for item in report_base:
            print(f"{item['word']:<15} | {item['status']}")
        print("_____________________________________")
    
        print("--- SMALL REPORT ---")
        for item in report_small:
            print(f"{item['word']:<15} | {item['status']}")
        print("_____________________________________")
    
        print(f"Scores -> Tiny: {score_tiny} | Base: {score_base} | Small: {score_small}")
    
    print("\n\n")
    print("__________________________________________________________________")



# Fuzzy keyword accuracy (%)
fuzzy_tiny_acc = sum(tiny_acc_fuzzy) / len(tiny_acc_fuzzy) * 100
fuzzy_base_acc = sum(base_acc_fuzzy) / len(base_acc_fuzzy) * 100
fuzzy_small_acc = sum(small_acc_fuzzy) / len(small_acc_fuzzy) * 100

# Shadowing Pass Rate (%)
threshold = 80
shadow_tiny_pass = [score >= threshold for score in tiny_acc_shadwing]
shadow_base_pass = [score >= threshold for score in base_acc_shadwing]
shadow_small_pass = [score >= threshold for score in small_acc_shadwing]

shadow_tiny_acc = sum(shadow_tiny_pass) / len(shadow_tiny_pass) * 100
shadow_base_acc = sum(shadow_base_pass) / len(shadow_base_pass) * 100
shadow_small_acc = sum(shadow_small_pass) / len(shadow_small_pass) * 100

# Shadowing Average Score
shadow_tiny_avg = sum(tiny_acc_shadwing) / len(tiny_acc_shadwing)
shadow_base_avg = sum(base_acc_shadwing) / len(base_acc_shadwing)
shadow_small_avg = sum(small_acc_shadwing) / len(small_acc_shadwing)

# Average Latency (Time)
avg_time_tiny = sum(tiny_latency) / len(tiny_latency)
avg_time_base = sum(base_latency) / len(base_latency)
avg_time_small = sum(small_latency) / len(small_latency)


print("\n=== FINAL RESULTS ===")
print(f"Fuzzy Accuracy - Tiny:  {fuzzy_tiny_acc:.2f}%")
print(f"Fuzzy Accuracy - Base:  {fuzzy_base_acc:.2f}%")
print(f"Fuzzy Accuracy - Small: {fuzzy_small_acc:.2f}%")
print("-" * 30)
print(f"Shadowing Accuracy  Tiny:  {shadow_tiny_acc:.2f}%")
print(f"Shadowing Accuracy  Base:  {shadow_base_acc:.2f}%")
print(f"Shadowing Accuracy Small: {shadow_small_acc:.2f}%")
print("-" * 30)
print(f"Shadowing Avg Score - Tiny:  {shadow_tiny_avg:.2f}")
print(f"Shadowing Avg Score - Base:  {shadow_base_avg:.2f}")
print(f"Shadowing Avg Score - Small: {shadow_small_avg:.2f}")
print("-" * 30)
print(f"Avg Latency (sec) - Tiny:  {avg_time_tiny:.4f}s")
print(f"Avg Latency (sec) - Base:  {avg_time_base:.4f}s")
print(f"Avg Latency (sec) - Small: {avg_time_small:.4f}s")

Target:        ['red', 'car']
Whisper Tiny:  ['red', 'color']  (Time: 2.208s)
Whisper Base:  ['red', 'car']  (Time: 1.353s)
Whisper Small: ['red', 'car']  (Time: 3.664s)



fuzzy match
Tiny:  False
Base:  True
Small: True


_____________________________________





__________________________________________________________________
Target:        ['one', 'plus', 'one', 'is', 'two']
Whisper Tiny:  ['one', 'plus', 'one', 'is', 'two']  (Time: 0.755s)
Whisper Base:  ['one', 'plus', 'one', 'is', 'two']  (Time: 1.413s)
Whisper Small: ['one', 'plus', 'one', 'is', 'two']  (Time: 3.877s)



shadowing match
--- TINY REPORT ---
one             | match
plus            | match
one             | match
is              | match
two             | match
_____________________________________
--- BASE REPORT ---
one             | match
plus            | match
one             | match
is              | match
two             | match
_____________________________________
--- SMALL REPORT ---
one             | 

## Fuzzy vs Shadowing Comparison

| Model | Fuzzy Accuracy | Shadowing Accuracy | Shadowing Avg. Score |
| :--- | :--- | :--- | :--- |
| **US Model** | 66.67% | 75.00% | 73.33% |
| **Small Model** | 75.00% | 66.67% | 67.92% |
| **Whisper Tiny** | 58.33% | 66.67% | 77.75% |
| **Whisper Base** | 75.00% | 83.33% | 86.08% |
| **Whisper Small** | **91.67%** | **91.67%** | **95.83%** |

---

## Separated Fuzzy & Shadowing Logic

| Model | Fuzzy Accuracy | Shadowing Accuracy | Shadowing Avg. Score |
| :--- | :--- | :--- | :--- |
| **US Model** | 57.14% | **100.00%** | 96.00% |
| **Small Model** | 71.43% | 80.00% | 83.00% |
| **Whisper Tiny** | 28.57% | **100.00%** | **100.00%** |
| **Whisper Base** | 57.14% | **100.00%** | **100.00%** |
| **Whisper Small** | **85.71%** | **100.00%** | **100.00%** |